<div style="padding:2.2rem;border-radius:18px;background:linear-gradient(135deg,#4c1d95,#0f766e);color:white;">
<p style="font-size:1.05rem;letter-spacing:.12em;text-transform:uppercase;margin:0;">D152 · Student Exercise</p>
<h1 style="font-size:3rem;margin:.5rem 0;">Change Data Capture with MySQL Triggers</h1>
<p style="font-size:1.35rem;margin:0;">Build an ordered change history for a QuickCart products table</p>
</div>

**You will design and implement the tables, triggers, tests, and validation queries yourself.**

# Exercise outcomes

By completing this exercise, you will be able to:

- Explain trigger-based **Change Data Capture (CDC)**
- Use `OLD` and `NEW` row images correctly
- Capture inserts, deletes, and both versions of an update
- Assign an ordered sequence number to every captured event
- Validate whether the change table reconstructs product history
- Recognize practical limitations of trigger-based CDC

# Scenario

QuickCart stores the current state of every product in `products`. Reporting and audit consumers also need an ordered record of changes.

Your task is to build this flow:

```text
Application change
       │
       ▼
products table ── MySQL triggers ──▶ products_ct table
 current state                         change history
```

`products` keeps only the latest row. `products_ct` stores immutable snapshots produced by the triggers.

# MySQL connection

Use your running MySQL instance:

| Setting | Classroom value |
|---|---|
| Host | `localhost` |
| Username | `root` |
| Password | `root` |

You may work in MySQL Workbench or the MySQL command-line client. Create a separate database for the exercise and select it before creating objects.

> These simple credentials are for the local classroom environment only. Do not use them for a shared or production server.

# Task 1 — Create the source table

Create a table named `products` with these requirements:

| Column | Requirement |
|---|---|
| `product_id` | Integer primary key |
| `name` | Required product name |
| `sku` | Required and unique |
| `price` | Exact decimal value; must not be negative |
| `qty` | Integer quantity; must not be negative |
| `created_at` | Defaults to the current timestamp |
| `updated_at` | Defaults to the current timestamp and refreshes when the row changes |

### Checkpoint
Use a metadata command to confirm the columns, keys, defaults, and constraints before continuing.

# Task 2 — Design `products_ct`

Create a change table named `products_ct`. It must contain:

- `seq`: a unique, automatically generated sequence number
- `op_code`: the operation code
- A snapshot of every business and timestamp column from `products`
- `captured_at`: the time the change record was captured

### Sequence hint
MySQL does not use standalone sequences in the same way as some databases. For this exercise, define `seq` as an integer type with **`AUTO_INCREMENT`**, and make it a key. Let MySQL generate it—triggers should not calculate `MAX(seq) + 1`.

> Do not make `product_id` unique in the change table. One product must be allowed to have many history rows.

# Operation-code contract

Your triggers must follow this mapping exactly:

| Code | Event | Snapshot to capture |
|---:|---|---|
| `1` | Delete | Row as it existed before deletion |
| `2` | Insert | Newly inserted row |
| `3` | Before update | Original row before values change |
| `4` | After update | New row after values change |

One update must therefore create **two consecutive history records**: first code `3`, then code `4`.

Add a constraint to `products_ct` if your MySQL version supports it, so only the four valid codes are accepted.

# `OLD` and `NEW` row images

A trigger can access values associated with the row that fired it:

| Trigger event | `OLD` available? | `NEW` available? | Use for this exercise |
|---|:---:|:---:|---|
| `INSERT` | No | Yes | Capture `NEW` with code `2` |
| `DELETE` | Yes | No | Capture `OLD` with code `1` |
| `UPDATE` before image | Yes | Yes | Capture `OLD` with code `3` |
| `UPDATE` after image | Yes | Yes | Capture `NEW` with code `4` |

Think of `OLD` and `NEW` as two row-shaped records. Access an individual value with the appropriate qualifier and column name.

# Non-runnable reference — insert trigger shape

The following uses an unrelated `inventory` table and placeholders. Adapt the idea; do not run it directly.

```sql
CREATE TRIGGER <insert_trigger_name>
AFTER INSERT ON inventory
FOR EACH ROW
BEGIN
    INSERT INTO inventory_history (<operation>, <snapshot_columns>)
    VALUES (<insert_code>, NEW.<column>, ...);
END;
```

### Think before implementing
Which source image exists after an insert? Should your trigger supply the sequence number, or let the table generate it?

# Non-runnable reference — delete trigger shape

```sql
CREATE TRIGGER <delete_trigger_name>
BEFORE DELETE ON inventory
FOR EACH ROW
BEGIN
    INSERT INTO inventory_history (<operation>, <snapshot_columns>)
    VALUES (<delete_code>, OLD.<column>, ...);
END;
```

### Think before implementing
After deletion, the source row no longer exists. Which row image preserves its last known values?

> A single-statement trigger may not require `BEGIN`, `END`, or delimiter changes. Multi-statement triggers normally do.

# Non-runnable reference — update trigger shapes

An update needs two triggers for this exercise:

```sql
CREATE TRIGGER <before_update_name>
BEFORE UPDATE ON inventory
FOR EACH ROW
    INSERT INTO inventory_history (...)
    VALUES (<before_code>, OLD.<column>, ...);

CREATE TRIGGER <after_update_name>
AFTER UPDATE ON inventory
FOR EACH ROW
    INSERT INTO inventory_history (...)
    VALUES (<after_code>, NEW.<column>, ...);
```

The first snapshot answers **what was replaced?** The second answers **what replaced it?**

# Task 3 — Implement four CDC triggers

Create one trigger for each requirement:

1. After inserting a product, capture `NEW` with code `2`.
2. Before deleting a product, capture `OLD` with code `1`.
3. Before updating a product, capture `OLD` with code `3`.
4. After updating a product, capture `NEW` with code `4`.

Each captured record must include a complete product snapshot. Do not manually supply `seq`; allow the change table to generate it.

### Checkpoint
Inspect database metadata and confirm that exactly four triggers exist on `products`, with the intended timing and event.

# Task 4 — Test the CDC flow

Perform these source-table actions in order:

1. Insert product A.
2. Insert product B.
3. Change product A's price and quantity in one update.
4. Change product B's name only.
5. Delete product A.

Before testing, predict the operation-code sequence. After testing, list `products_ct` in ascending `seq` order and compare it with your prediction.

### Expected history size
Two inserts create 2 rows, two updates create 4 rows, and one delete creates 1 row: **7 change rows in total**.

# Validation checklist

Your implementation is complete when all answers are **yes**:

- Does every inserted product produce exactly one code `2` record?
- Does every deleted product produce exactly one code `1` record?
- Does every update produce code `3` followed by code `4`?
- Does code `3` contain the original values?
- Does code `4` contain the updated values?
- Are sequence values unique and increasing?
- Do source timestamps appear in the captured snapshots?
- Does `captured_at` record when each history row was written?
- Can one product ID appear many times in `products_ct`?
- Does an invalid operation code fail?

# Reflection and extension challenges

### Explain in your own words

- Why is `MAX(seq) + 1` unsafe when sessions write concurrently?
- Why does an update create two records in this design?
- What happens to CDC records if the source transaction rolls back?
- What information is missing about the user or application that made a change?
- How might large trigger-based history tables affect write performance?

### Optional extensions

- Capture the database user responsible for the change.
- Add a shared change identifier to pair pre-update and post-update rows.
- Prevent updates and deletes on `products_ct`.
- Compare this design with MySQL binary-log-based CDC.